In [0]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

RAW_PATH = "/Volumes/workspace/default/insure_data/raw"
BRONZE_PATH = "/Volumes/workspace/default/insure_data/bronze"

TABLES = [
    # "agent",
    # "agent_policy",
    # "customer",
    # "customer_policy",
    # "customer_role_master",
    "money_in_dtl",
    "policy",
    # "product_commission_rule",
    # "product_master"
]


def ingest_to_bronze(table_name):

    try:

        print(f"\n========== {table_name.upper()} ==========")

        source_path = f"{RAW_PATH}/{table_name}.parquet"
        target_path = f"{BRONZE_PATH}/{table_name}"

        df = (
            spark.read
            .parquet(source_path)
        )

        bronze_df = (
            df
            .withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("ingestion_date", F.current_date())
        )

        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .save(target_path)
        )

        print(f"Rows Loaded : {bronze_df.count()}")

    except AnalysisException as e:
        print(f"Error loading {table_name}")
        print(e)

    except Exception as e:
        print(e)


for table in TABLES:
    ingest_to_bronze(table)

In [0]:
display(
    spark.read.parquet(f"{BRONZE_PATH}/agent")
)
# same way all tables

In [0]:
spark.read.parquet(f"{BRONZE_PATH}/agent").count()

In [0]:
spark.read.parquet(f"{BRONZE_PATH}/agent").printSchema()